# 03 - Baseline Models: Linear Regression

**Goal:** Establish baseline performance so we know what "good" looks like.

We start with the dumbest possible models (predict the mean, predict last week), then build up to linear regression. Each model must beat the previous one to justify its complexity.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'prediction'))

from prediction.src.evaluation import evaluate_model, plot_predictions_vs_actual, plot_residuals

DATA_DIR = os.path.join(PROJECT_ROOT, 'prediction', 'data')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

# Load data
df = pd.read_parquet(os.path.join(DATA_DIR, 'features_all.parquet'))
with open(os.path.join(DATA_DIR, 'feature_columns.json')) as f:
    FEATURE_COLS = json.load(f)

TARGET = 'total_points'
print(f'Dataset: {df.shape[0]:,} rows, {len(FEATURE_COLS)} features')
print(f'Gameweek range: {df["round"].min()} - {df["round"].max()}')

## 1. Temporal Train/Test Split

**Critical:** We split by time, not randomly. Random splitting would leak future data into training.

We use ~75% of gameweeks for training, the rest for testing. The exact split point depends on available data.

In [ ]:
max_gw = df['round'].max()
split_gw = int(max_gw * 0.75)

train = df[df['round'] <= split_gw]
test = df[df['round'] > split_gw]

X_train = train[FEATURE_COLS]
y_train = train[TARGET]
X_test = test[FEATURE_COLS]
y_test = test[TARGET]

print(f'Split at GW {split_gw}')
print(f'Train: GW 1-{split_gw} | {len(train):,} rows')
print(f'Test:  GW {split_gw+1}-{max_gw} | {len(test):,} rows')

## 2. Dumb Baselines

These are the floor. Any real model must beat these.

1. **Predict mean** — always predict the training set average. This is the simplest possible model.
2. **Predict last week** — predict whatever the player scored last GW. This captures recent form naively.

In [ ]:
results = {}

# Baseline 1: Predict mean
mean_pred = np.full(len(y_test), y_train.mean())
results['Predict Mean'] = evaluate_model(y_test, mean_pred)

# Baseline 2: Predict last week's points (pts_rolling_1 is essentially shift(1))
# We use pts_rolling_3 as a proxy since we built 3-GW rolling, not 1-GW
# A better proxy: the season_avg_pts is the expanding mean of prior GWs
last_week_pred = test['pts_rolling_3'].values  # 3GW rolling average as "recent form"
results['Recent Form (3GW avg)'] = evaluate_model(y_test, last_week_pred)

print('Baseline Results:')
print(f'{"Model":<25s} {"MAE":>8s} {"RMSE":>8s} {"R2":>8s}')
print('-' * 51)
for name, metrics in results.items():
    print(f'{name:<25s} {metrics["MAE"]:8.3f} {metrics["RMSE"]:8.3f} {metrics["R2"]:8.3f}')

## 3. Linear Regression

Linear regression finds the best straight-line relationship between features and target. It assumes the relationship is linear and additive: `points = w1*feature1 + w2*feature2 + ... + bias`

We scale features first because linear regression is sensitive to feature magnitudes — a feature ranging 0-1000 would dominate one ranging 0-1.

In [ ]:
# Scale features (fit on train, transform both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Predict
lr_pred_train = lr.predict(X_train_scaled)
lr_pred_test = lr.predict(X_test_scaled)

results['Linear Regression'] = evaluate_model(y_test, lr_pred_test)
train_metrics = evaluate_model(y_train, lr_pred_train)

print('Linear Regression:')
print(f'  Train — MAE: {train_metrics["MAE"]:.3f}, R2: {train_metrics["R2"]:.3f}')
print(f'  Test  — MAE: {results["Linear Regression"]["MAE"]:.3f}, R2: {results["Linear Regression"]["R2"]:.3f}')
print(f'\n  Gap between train and test R2: {train_metrics["R2"] - results["Linear Regression"]["R2"]:.3f}')
print('  (Large gap = overfitting, small gap = generalizes well)')

## 4. Coefficient Inspection

In linear regression, each feature gets a **coefficient** (weight). Positive = more of this feature means higher predicted points. The magnitude tells you how important it is (after scaling).

In [ ]:
coef_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': lr.coef_
}).sort_values('coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, max(8, len(FEATURE_COLS) * 0.3)))

# Show top 15 and bottom 5
display_coefs = pd.concat([coef_df.head(15), coef_df.tail(5)])
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in display_coefs['coefficient']]
ax.barh(range(len(display_coefs)), display_coefs['coefficient'], color=colors)
ax.set_yticks(range(len(display_coefs)))
ax.set_yticklabels(display_coefs['feature'])
ax.set_xlabel('Coefficient (scaled)')
ax.set_title('Linear Regression Coefficients')
ax.axvline(0, color='black', linewidth=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Predictions vs Actual + Residual Analysis

In [ ]:
fig = plot_predictions_vs_actual(y_test, lr_pred_test, 'Linear Regression: Predicted vs Actual')
plt.show()

In [ ]:
fig = plot_residuals(y_test, lr_pred_test, 'Linear Regression')
plt.show()

**Residual patterns to look for:**
- **Funnel shape** (residuals spread increases with predicted value) → heteroscedasticity, model struggles with high scorers
- **Systematic bias** (mean residual != 0) → model consistently over/under-predicts
- **Clusters** → distinct subgroups the model treats identically (e.g., positions)

## 6. Per-Position Baseline

In [ ]:
print(f'{"Position":<15s} {"MAE":>8s} {"RMSE":>8s} {"R2":>8s} {"Count":>8s}')
print('-' * 55)

position_results = {}
for pos in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    mask = test['Position'] == pos
    if mask.sum() == 0:
        continue
    metrics = evaluate_model(y_test[mask], lr_pred_test[mask])
    position_results[pos] = metrics
    print(f'{pos:<15s} {metrics["MAE"]:8.3f} {metrics["RMSE"]:8.3f} {metrics["R2"]:8.3f} {mask.sum():8d}')

## 7. Results Summary

In [ ]:
# Save results for comparison in Phase 6
results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'

print('\nAll Models Comparison:')
print(results_df.round(3).to_string())

# Save to disk
results_df.to_csv(os.path.join(DATA_DIR, 'results_phase3.csv'))
print(f'\nResults saved to {os.path.join(DATA_DIR, "results_phase3.csv")}')

## Summary

**Baselines established:**
- Predict Mean: the absolute floor
- Recent Form (3GW avg): simple but surprisingly competitive
- Linear Regression: uses all features but assumes linear relationships

**Key observations:**
- R-squared is expected to be low (0.05-0.15) — FPL points are inherently noisy
- Coefficient analysis shows which features linear regression values most
- Residual patterns suggest where non-linear models might improve

Next: `04_tree_models.ipynb` — tree-based models can capture non-linear patterns.